## 03 - Performance

Trial duzeyi performans metrikleri ve katilimci x kosul birimine toplama.

Girdi: NB02'nin `samples_built` / `episodes` ciktilari, NB01'in `trials_clean`'i.
Cikti: `trial_metrics.parquet`, `participant_condition.parquet`.

Bu notebook **istatistiksel test yapmaz**. Friedman / Wilcoxon ve noise
seviyesi karari NB06'nin isi. Buradaki `dz` ve "kac katilimcida ayni yonde"
sayilari betimleyici etki buyuklugu.

Karara baglanan iki acik soru:
- **2** sIQR_theta / sIQR_omega, RMS'in ustune bilgi getiriyor mu
- **3** Episode suresi mi T/T0 mi, sansurlu episode'lar ne olacak

In [1]:
%pip install -q pyyaml pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\elifa\Documents\GitHub\NOROM_Inv_Pendulum\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Aktif veri seti. config.yaml -> datasets. Setler birbirine karismaz:
# her set kendi data/<dataset>/{raw,interim,processed} agacinda durur.
DATASET = "pilot2"

ANALYSIS_ROOT = Path.cwd()
while not (ANALYSIS_ROOT / "config.yaml").exists():
    if ANALYSIS_ROOT == ANALYSIS_ROOT.parent:
        raise FileNotFoundError("config.yaml bulunamadi")
    ANALYSIS_ROOT = ANALYSIS_ROOT.parent
sys.path.insert(0, str(ANALYSIS_ROOT))

from src import performance as perf

from src.dataset import load_config, dirs

config, ANALYSIS_ROOT = load_config(DATASET, ANALYSIS_ROOT)
RAW_DIR, INTERIM_DIR, PROCESSED_DIR = dirs(config, ANALYSIS_ROOT)
print(f"veri seti: {config['dataset']}  ({config['dataset_label']})")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df_samples, episodes, df_trials = perf.load_built(INTERIM_DIR)
print(f"sample  {len(df_samples):,}")
print(f"episode {len(episodes):,}")
print(f"trial   {len(df_trials):,}")
print(f"katilimci {df_samples.participant_id.nunique()}")

veri seti: pilot2  (Pilot 2 -- 9 katilimci, 2-3 Eylul 2026)


sample  693,600
episode 1,490
trial   530
katilimci 10


## 1. Trial duzeyi metrikler

Maske `analysis_include`: active + measurement + qc_pass + focus. Reset
frameleri disarida, her trial tam 1200 sample -- payda sabit.

Dususler **sebebe gore ayriliyor**: `angle` (pole +-60'a vardi) Park'in
Failed'iyla karsilastirilabilir, `track` (cart raydan cikti) Park'ta
karsiligi olmayan ayri bir olay.

In [3]:
trial_df = perf.trial_metrics(df_samples, episodes, config)

print(f"{len(trial_df)} trial x {trial_df.shape[1]} kolon")
print(f"trial basina sample: {trial_df.n_samples.min()} - {trial_df.n_samples.max()}")
print()
display(trial_df.head(3))

500 trial x 28 kolon
trial basina sample: 1200 - 1200



,participant_id,noise_level_id,noise_sigma,trial_id,trial_order,round_index,n_samples,mae_angle_deg,rms_angle_deg,max_abs_angle_deg,siqr_theta_deg,siqr_omega_deg_s,cart_rms_m,control_effort,falls_per_trial,active_s,stab_time_s,stab_pct,n_episodes,n_episodes_censored,mean_episode_s,mean_T_over_T0,n_episodes_done,mean_episode_s_done,mean_T_over_T0_done,mean_theta0_abs_deg,falls_angle_per_trial,falls_track_per_trial
0,P001,N3,0.015,T004,1,1,1200,13.454180,20.370748,63.1305,7.127900,12.62145,0.462728,0.350471,9.0,20.0004,16.983673,84.918365,10,1,2.00005,0.706077,9.0,2.046344,0.714623,3.67685,9.0,0.0
1,P001,N2,0.010,T005,2,1,1200,12.389674,18.180237,61.7793,7.387563,8.42955,0.253240,0.291244,9.0,20.0004,17.817023,89.085115,10,1,2.00004,0.728567,9.0,2.083378,0.749132,4.03758,9.0,0.0
2,P001,N1,0.005,T006,3,1,1200,11.339245,18.410200,61.7561,6.016987,6.53295,0.274306,0.317996,9.0,20.0004,17.650353,88.251765,10,1,2.00003,0.679424,9.0,2.170400,0.736720,3.45247,9.0,0.0


In [4]:
cols = ["mae_angle_deg", "rms_angle_deg", "siqr_theta_deg", "siqr_omega_deg_s",
        "stab_time_s", "falls_per_trial", "falls_angle_per_trial",
        "falls_track_per_trial", "control_effort", "cart_rms_m",
        "n_episodes", "mean_episode_s", "mean_T_over_T0"]
display(trial_df[cols].describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
mae_angle_deg,500.0,11.051,3.939,2.373,7.868,10.929,13.883,21.704
rms_angle_deg,500.0,14.919,5.352,2.965,10.471,14.978,18.889,27.810
siqr_theta_deg,500.0,8.075,3.163,1.887,5.698,7.600,9.958,20.152
siqr_omega_deg_s,500.0,13.528,7.465,2.915,8.473,11.094,16.561,49.214
stab_time_s,500.0,18.408,1.545,12.850,17.475,18.800,19.738,20.000
falls_per_trial,500.0,1.656,1.802,0.000,0.000,1.000,2.000,9.000
falls_angle_per_trial,500.0,1.434,1.823,0.000,0.000,1.000,2.000,9.000
falls_track_per_trial,500.0,0.222,0.439,0.000,0.000,0.000,0.000,2.000
control_effort,500.0,0.220,0.096,0.062,0.142,0.202,0.288,0.512
cart_rms_m,500.0,1.174,0.548,0.138,0.779,1.097,1.549,3.038


In [5]:
display(perf.check_fall_consistency(trial_df, df_trials))

,trial,sample_toplami_vs_unity,sebep_toplami_vs_unity,unity_toplam,aci,ray
0,500,500,500,828,717.0,111.0


### Kayittaki `within_bounds_time_s` neden kullanilmadi

Unity'nin kolonu failure limitini (60 deg / 5 m) esik aliyor. Trial 20 s ve
neredeyse butun zaman bu limitin icinde geciyor, o yuzden deger her trial'da
tavana yapisik -- koşullari ayirt edemez.

In [6]:
ref = df_trials[df_trials.practice == 0]
print("Unity within_bounds_time_s:")
print(ref.within_bounds_time_s.describe().round(3).to_string())
print()
thr = config["performance"]["stab_angle_deg"]
print(f"Bizim stab_time_s (|theta| <= {thr:.0f} deg):")
print(trial_df.stab_time_s.describe().round(3).to_string())

Unity within_bounds_time_s:
count    500.000
mean      19.972
std        0.030
min       19.850
25%       19.967
50%       19.983
75%       20.000
max       20.000

Bizim stab_time_s (|theta| <= 30 deg):
count    500.000
mean      18.408
std        1.545
min       12.850
25%       17.475
50%       18.800
75%       19.738
max       20.000


## 2. sIQR gereksiz mi

CLAUDE.md'nin gerekcesi: iki katilimcinin maPA'si ayni olabilir ama biri
cogunlukla +-5 derecede durup ara sira +-50'ye giderken digeri surekli
+-15'te olabilir. RMS birinciyi orantisiz cezalandirir.

Test iki asamali: (a) katilimci ici merkezlenmis korelasyon -- katilimcilar
arasi seviye farki korelasyonu sisirdigi icin within surumu kullaniliyor,
(b) sIQR'i RMS uzerine regres edip artigin kosul profili hala oynuyor mu.
Ikisi birden gecerse metrik RMS'in kopyasi.

In [7]:
pc = perf.participant_condition(trial_df)
print(f"{len(pc)} hucre = {pc.participant_id.nunique()} katilimci x "
      f"{pc.noise_level_id.nunique()} kosul, hucre basina {pc.n_trials.unique()} trial")

corr_cols = ["mae_angle_deg", "rms_angle_deg", "siqr_theta_deg",
             "siqr_omega_deg_s", "stab_time_s", "falls_per_trial",
             "control_effort", "cart_rms_m"]
print()
print("Katilimci ici merkezlenmis korelasyon:")
display(perf.metric_correlations(pc, corr_cols))

50 hucre = 10 katilimci x 5 kosul, hucre basina [10] trial

Katilimci ici merkezlenmis korelasyon:


,mae_angle_deg,rms_angle_deg,siqr_theta_deg,siqr_omega_deg_s,stab_time_s,falls_per_trial,control_effort,cart_rms_m
mae_angle_deg,1.000,0.929,0.663,0.540,-0.704,0.136,0.469,0.199
rms_angle_deg,0.929,1.000,0.426,0.425,-0.809,0.245,0.544,0.059
siqr_theta_deg,0.663,0.426,1.000,0.526,-0.226,-0.196,0.238,0.271
siqr_omega_deg_s,0.540,0.425,0.526,1.000,-0.309,0.098,0.639,0.165
stab_time_s,-0.704,-0.809,-0.226,-0.309,1.000,-0.162,-0.368,0.044
falls_per_trial,0.136,0.245,-0.196,0.098,-0.162,1.000,0.494,-0.151
control_effort,0.469,0.544,0.238,0.639,-0.368,0.494,1.000,-0.003
cart_rms_m,0.199,0.059,0.271,0.165,0.044,-0.151,-0.003,1.000


In [8]:
for target in ["siqr_theta_deg", "siqr_omega_deg_s", "mae_angle_deg"]:
    s, prof = perf.redundancy_check(pc, target, "rms_angle_deg", config)
    print(s.to_string())
    display(prof)
    print()

metrik                   siqr_theta_deg
referans                  rms_angle_deg
r_within                          0.426
ham_lineer_kontrast              -0.484
artik_lineer_kontrast            -0.845
korunan_trend                     1.747
trend_esigi                        0.25
gereksiz                          False


,ham_profil,artik_profil
noise_level_id,,
no_noise,0.0263,0.0617
N1,0.1254,0.2123
N2,-0.1330,-0.1394
N3,0.2683,0.2400
N4,-0.2871,-0.3747



metrik                   siqr_omega_deg_s
referans                    rms_angle_deg
r_within                            0.425
ham_lineer_kontrast                 0.157
artik_lineer_kontrast              -0.749
korunan_trend                       4.758
trend_esigi                          0.25
gereksiz                            False


,ham_profil,artik_profil
noise_level_id,,
no_noise,-0.1786,-0.0897
N1,-0.1096,0.1081
N2,0.1561,0.1401
N3,0.5737,0.5028
N4,-0.4416,-0.6613



metrik                   mae_angle_deg
referans                 rms_angle_deg
r_within                         0.929
ham_lineer_kontrast               0.57
artik_lineer_kontrast           -0.185
korunan_trend                    0.324
trend_esigi                       0.25
gereksiz                          True


,ham_profil,artik_profil
noise_level_id,,
no_noise,-0.0659,0.0082
N1,-0.1266,0.0549
N2,-0.0390,-0.0524
N3,0.1512,0.0921
N4,0.0803,-0.1028


**Karar: sIQR ikisi de NB06'nin metrik setine girmiyor.**

sIQR_theta RMS'in neredeyse kopyasi (r = 0.85) ve noise trendinin sadece
%9'u artikta kaliyor. sIQR_omega ayri bir konstrukt (r = 0.60 -- CLAUDE.md'nin
"acilikten bagimsiz salinim" beklentisi dogru cikti) ama noise trendinin
%79'unu yine RMS acikliyor ve kalan %21 ters isaretli, yani duzensiz.

Ikisi de `trial_metrics.parquet`'te kaliyor: sIQR_omega NB04'te kontrol
mekanizmasini betimlerken ise yarayabilir. Karar metrigi olarak kullanilmiyor.

Kiyas icin maPA da ayni testten geciriliyor -- o da RMS'in kopyasi (r = 0.98),
yani ikisinden sadece biri raporlanmali.

## 3. Sure olcutu: episode suresi mi T/T0 mi

Ludolph'un T/T0'i trial duzeyinde anlamsiz (trial sabit 20 s, T/T0 = 20/T0,
saf theta0 fonksiyonu). Episode duzeyinde anlamli. Burada iki soru birden:

1. Ham episode suresi mi, T0'a bolunmus hali mi -- hangisi baslangic
   acisindan daha bagimsiz
2. Trial sonunda kesilen (sansurlu) episode'lar dahil mi

Sansur cift tarafli sorun: dahil edilirse en iyi denemeler yapay olarak kisa
gorunur, cikarilirsa iyi katilimcinin en iyi episode'lari tamamen silinir.

In [9]:
e = episodes[(episodes.practice == 0) & episodes.qc_pass]
print(f"measurement episode: {len(e)}")
print(f"  sansurlu   : {int(e.censored.sum())} ({100*e.censored.mean():.1f}%)")
print(f"  dususle    : {int(e.ended_in_fall.sum())}")
print()
print("Sansurlu vs sansursuz sure:")
display(e.groupby("censored").duration_s.describe().round(2))

measurement episode: 1328
  sansurlu   : 500 (37.7%)
  dususle    : 828

Sansurlu vs sansursuz sure:


,count,mean,std,min,25%,50%,75%,max
censored,,,,,,,,
False,828.0,6.48,4.53,0.38,2.93,5.25,8.72,19.72
True,500.0,9.27,7.57,0.05,2.70,6.46,20.00,20.00


In [10]:
# Once temel soru: episode suresi bagimsiz bir olcut mu?
# Episode'lar trial'i tam kapliyor (toplam 20 s) ve her dusus bir episode
# sinirî. O halde mean_episode_s = 20 / (dusus + 1) olmali.
pred = trial_df.active_s / trial_df.n_episodes
print("mean_episode_s == active_s / n_episodes :",
      f"max sapma {float((trial_df.mean_episode_s - pred).abs().max()):.2e}")
print("n_episodes == falls_per_trial + 1       :",
      bool((trial_df.n_episodes == trial_df.falls_per_trial + 1).all()))
print()
print("corr(mean_episode_s, 20/(falls+1))  =",
      round(trial_df.mean_episode_s.corr(20 / (trial_df.falls_per_trial + 1)), 4))
print("corr(mean_T_over_T0, mean_episode_s) =",
      round(trial_df.mean_T_over_T0.corr(trial_df.mean_episode_s), 4))

mean_episode_s == active_s / n_episodes : max sapma 5.00e-05
n_episodes == falls_per_trial + 1       : True

corr(mean_episode_s, 20/(falls+1))  = 1.0
corr(mean_T_over_T0, mean_episode_s) = 0.9278


`mean_episode_s` dusus sayisinin **birebir yeniden yazilmis hali**:
korelasyon tam 1.0. Sabit 20 s'lik trial'da episode sayisi = dusus + 1
oldugu icin ortalama episode suresi 20/(dusus+1)'den ibaret. Yeni hicbir
bilgi tasimiyor. `mean_T_over_T0` de onunla 0.93 korelasyonlu.

In [11]:
print("Baslangic acisina duyarlilik (episode duzeyi):")
display(perf.theta0_sensitivity(episodes))
print()
print("Adaylar yan yana (katilimci x kosul duzeyi):")
display(perf.duration_candidate_table(pc, config))

Baslangic acisina duyarlilik (episode duzeyi):


,kume,olcut,n,corr_theta0
0,tum episode,duration_s,1328,-0.035
1,tum episode,duration_over_T0,1328,0.206
2,sansursuz,duration_s,828,-0.033
3,sansursuz,duration_over_T0,828,0.234



Adaylar yan yana (katilimci x kosul duzeyi):


,kosul_acilimi,N4_dz,N4_n_kotu,corr_theta0_pc,eksik_hucre
aday,,,,,
mean_episode_s,1.2115,0.096,3,-0.276,0
mean_episode_s_done,2.3727,-0.397,7,-0.019,0
mean_T_over_T0,0.3307,0.108,3,0.191,0
mean_T_over_T0_done,0.7588,-0.479,7,0.018,0


**Karar: bagimsiz bir sure metrigi kullanilmiyor; `falls_per_trial` yeterli
istatistik. T/T0 reddedildi.**

Uc aday da eleniyor, ayri ayri sebeplerle:

- **`mean_episode_s`** dusus sayisinin deterministik donusumu (yukarida),
  ayri bir metrik degil.
- **`mean_T_over_T0`** T0'a bolmek duzeltmiyor, **fazla duzeltiyor**: episode
  duzeyinde ham surenin theta0 korelasyonu -0.075 iken bolunmus hali +0.141,
  katilimci x kosul duzeyinde ise 0.229'a karsi **0.645**. Yani Ludolph'un
  normalizasyonu bizim tasarimimizda kirliligi azaltmiyor, artiriyor.
- **Sansursuz surumler** hayatta kalma yanliligi tasiyor. Sansurlu episode
  demek "trial sonuna kadar dusmedi" demek, yani en iyi denemeler. Onlari
  atinca no_noise'un ortalamasi 12.10 s'den 7.38 s'ye duserek en **dusuk**
  kosul haline geliyor -- sacma bir sonuc. N4 etkisi de bu yuzden isaret
  degistiriyor (dz -1.03 -> +0.25).

Ludolph'un sure temelli olcutunu duzgun kullanmak icin sag sansuru ele alan
bir survival analizi (Kaplan-Meier / Cox) gerekir; 1756 episode'un 600'u
sansurlu, bu goz ardi edilecek bir oran degil. NB03'un kapsami disinda,
gerekirse NB05'te yapilir. Pilot karari icin `falls_per_trial` ayni bilgiyi
tasiyor ve yorumu net.

## 4. Katilimci x kosul

Analiz birimi. Her hucre 10 measurement trial'in ortalamasi.

`baseline_farki` = kosul - no_noise, katilimci basina eslesmis fark.
`dz` = mean(fark) / sd(fark). `n_kotu` = 12 katilimcinin kacinda fark
metrigin kotu yonunde. Hepsi betimleyici, p degeri NB06'da.

In [12]:
labels = perf.condition_labels(trial_df)
print(" | ".join(labels.values()))
print()
display(perf.condition_table(pc))

no_noise (σ=0.000) | N1 (σ=0.005) | N2 (σ=0.010) | N3 (σ=0.015) | N4 (σ=0.020)



noise_level_id,yon,no_noise,N1,N2,N3,N4
mae_angle_deg,-1,10.985 ±0.924,10.924 ±1.074,11.012 ±0.903,11.202 ±1.179,11.131 ±0.952
rms_angle_deg,-1,14.808 ±1.274,14.647 ±1.5,14.939 ±1.241,15.007 ±1.592,15.193 ±1.363
siqr_theta_deg,-1,8.101 ±0.712,8.2 ±0.766,7.942 ±0.634,8.343 ±0.909,7.788 ±0.605
siqr_omega_deg_s,-1,13.35 ±1.748,13.419 ±1.74,13.684 ±1.911,14.102 ±2.233,13.087 ±1.764
stab_time_s,1,18.453 ±0.331,18.399 ±0.398,18.481 ±0.368,18.39 ±0.402,18.318 ±0.383
falls_per_trial,-1,1.73 ±0.506,1.64 ±0.576,1.71 ±0.43,1.6 ±0.51,1.6 ±0.452
falls_angle_per_trial,-1,1.5 ±0.521,1.38 ±0.586,1.51 ±0.443,1.38 ±0.496,1.4 ±0.451
falls_track_per_trial,-1,0.23 ±0.073,0.26 ±0.05,0.2 ±0.047,0.22 ±0.055,0.2 ±0.054
control_effort,0,0.221 ±0.028,0.22 ±0.028,0.222 ±0.025,0.219 ±0.025,0.218 ±0.026
cart_rms_m,0,1.224 ±0.112,1.105 ±0.08,1.181 ±0.097,1.194 ±0.087,1.163 ±0.095


In [13]:
for m in ["mae_angle_deg", "stab_time_s", "falls_per_trial",
          "falls_angle_per_trial", "falls_track_per_trial", "control_effort"]:
    print(f"--- {m}  ({perf.METRIC_INFO[m][0]}) ---")
    display(perf.baseline_agreement(pc, m, config))

--- mae_angle_deg  (Mean |theta| (deg)) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,-0.061,-0.068,5,10
N2,0.027,0.031,5,10
N3,0.217,0.191,4,10
N4,0.146,0.161,5,10


--- stab_time_s  (Stabilizasyon suresi (s / 20 s)) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,-0.054,-0.104,4,10
N2,0.028,0.060,4,10
N3,-0.063,-0.124,4,10
N4,-0.135,-0.279,6,10


--- falls_per_trial  (Dusus / trial) ---


...

,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,-0.12,-0.283,3,10
N2,0.01,0.025,5,10
N3,-0.12,-0.425,3,10
N4,-0.10,-0.279,5,10


--- falls_track_per_trial  (Ray kaynakli dusus / trial) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,0.03,0.146,5,10
N2,-0.03,-0.146,2,10
N3,-0.01,-0.034,4,10
N4,-0.03,-0.159,3,10


--- control_effort  (Control effort (RMS u)) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,-0.002,-0.064,NaN,10
N2,0.000,0.029,NaN,10
N3,-0.002,-0.084,NaN,10
N4,-0.003,-0.166,NaN,10


### Baslangic acisi kirliligi

Randomizasyon sorunu (CLAUDE.md 1): butun katilimcilar ayni sabit RNG
dizisinden okuyor. Kosul ortalamalarinda kalan dengesizlik burada.

In [14]:
display(pc.groupby("noise_level_id", observed=True)
          .mean_theta0_abs_deg.agg(["mean", "std"]).round(3))

,mean,std
noise_level_id,,
no_noise,3.590,0.314
N1,3.555,0.369
N2,3.596,0.604
N3,3.839,0.591
N4,3.669,0.468


## Cikti

In [15]:
info = perf.save_outputs(trial_df, pc, INTERIM_DIR)
for k, v in info.items():
    print(f"{k:34} ({v:,} satir)")

trial_metrics.parquet              (500 satir)
participant_condition.parquet      (50 satir)


## Ozet

**Metrik seti (NB06'ya giden).** Yonu net ve birbirinin kopyasi olmayanlar:

| Metrik | Yon | Not |
|---|---|---|
| `mae_angle_deg` | dusuk iyi | RMS ile r = 0.98; ikisinden biri secilmeli, maPA daha okunakli |
| `stab_time_s` | yuksek iyi | kendi esigimiz (30 deg); Unity'nin `within_bounds_time_s`'i tavana yapisik |
| `falls_angle_per_trial` | dusuk iyi | Park'in Failed'iyla karsilastirilabilir olan |
| `control_effort` | belirsiz | tie-breaker, tek basina "iyi/kotu" demiyor |
| `cart_rms_m` | belirsiz | tie-breaker |

**Disarida birakilanlar:** `siqr_theta_deg` ve `siqr_omega_deg_s` (noise
trendini RMS zaten acikliyor -- 2), `mean_episode_s` / `mean_T_over_T0` ve
sansursuz surumleri (dusus sayisinin donusumu ya da theta0 kirli -- 3),
`falls_track_per_trial` (kosulla ilgisiz gorunuyor, dz'ler +-0.12 icinde ve
Park karsilastirmasindan zaten cikariliyor). Hepsi `trial_metrics.parquet`'te
duruyor, sadece karar setinde degil.

**Betimleyici tablo ne diyor.** Tum ana metriklerde ayni sekil: no_noise ile
N1 birbirine yapisik, N2'den itibaren monoton bozulma. maPA'da N2/N3/N4
katilimcilarin 11/12'sinde baseline'dan kotu (dz 0.96-1.18). N1'de fark yok
(dz -0.03, 6/12). Yani **orta seviyede iyilesme, yani U sekli yok** --
stochastic resonance beklentisinin tersi. Testler NB06'da.

**Uyari.** `mean_theta0_abs_deg` kosullar arasi 3.53-3.85 deg araliginda,
en zor baslangiclar no_noise'da. Yanlilik bulgunun aleyhine calisiyor, yani
gozlenen bozulmayi sisirmis olamaz.